# One-Class SVM para Autenticidad de Billetes

Entrena un clasificador de una sola clase (autentico) usando SVM con kernel RBF.
Sin necesidad de billetes falsos. Cualquier feature que se aleje del hiperplano aprendido
se clasifica como FALSA.

---
## Flujo
1. Cargar y alinear crops con ORB+FLANN+Homografia
2. Extraer vector mixto 64d (histograma + laplaciano)
3. Entrenar One-Class SVM (OpenCV)
4. Evaluar y exportar a XML
---

In [23]:
import os, sys
import numpy as np
import cv2
from pathlib import Path
from collections import Counter

print("OpenCV:", cv2.__version__)

OpenCV: 4.13.0


---
## 2. Configuracion
---

In [24]:
SEGMENTS_DIR = Path("../Data/raw/imagenes_ir/segmentos_ir_billetes")
TEMPLATES_DIR = Path("../rpi_deployment/templates")
OUTPUT_MODEL = Path("../rpi_deployment/ocsvm.xml")

TEMPLATE_SIZE = (100, 100)
HIST_BINS = 32
LAPLACIAN_BINS = 32

ORB_FEATURES = 2000
FLANN_CHECKS = 100
RANSAC_THRESHOLD = 5.0
MATCH_RATIO = 0.75

# One-Class SVM params
SVM_NU = 0.05       # fraccion esperada de outliers (0.01-0.1)
SVM_GAMMA = 'auto'  # 'auto' = 1/n_features

print(f"Templates: {TEMPLATES_DIR}")
print(f"Dataset: {SEGMENTS_DIR}")

Templates: ../rpi_deployment/templates
Dataset: ../Data/raw/imagenes_ir/segmentos_ir_billetes


---
## 3. Alineador Multi-Template
---

In [25]:
class MultiTemplateAligner:
    def __init__(self, templates_dir):
        self.templates = {}
        self.template_kp = {}
        self.template_des = {}
        self.orb = cv2.ORB_create(nfeatures=ORB_FEATURES, scaleFactor=1.2, nlevels=1, edgeThreshold=3)
        ip = dict(algorithm=6, table_number=12, key_size=20, multi_probe_level=2)
        sp = dict(checks=FLANN_CHECKS)
        self.flann = cv2.FlannBasedMatcher(ip, sp)
        
        for path in sorted(TEMPLATES_DIR.glob("*.png")):
            ft = path.stem
            img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if img is None: continue
            img = cv2.resize(img, TEMPLATE_SIZE)
            self.templates[ft] = img
            kp, des = self.orb.detectAndCompute(img, None)
            self.template_kp[ft] = kp if kp else []
            self.template_des[ft] = des
        print(f"[ALIGNER] {len(self.templates)} templates")
    
    def align(self, img, feature_type):
        if feature_type not in self.templates or img is None or img.size == 0:
            return None
        ikp, ides = self.orb.detectAndCompute(img, None)
        if ides is not None and len(ides) >= 4:
            try:
                td = self.template_des.get(feature_type)
                if td is not None and len(td) >= 4:
                    m = self.flann.knnMatch(td, ides, k=2)
                    g = [a for a, b in m if a.distance < MATCH_RATIO * b.distance]
                    if len(g) >= 4:
                        sp_pts = np.float32([self.template_kp[feature_type][a.queryIdx].pt for a in g]).reshape(-1, 1, 2)
                        dp_pts = np.float32([ikp[a.trainIdx].pt for a in g]).reshape(-1, 1, 2)
                        H, _ = cv2.findHomography(dp_pts, sp_pts, cv2.RANSAC, RANSAC_THRESHOLD)
                        if H is not None:
                            return cv2.warpPerspective(img, H, TEMPLATE_SIZE, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
            except: pass
        return cv2.resize(img, TEMPLATE_SIZE)

aligner = MultiTemplateAligner(TEMPLATES_DIR)

[ALIGNER] 21 templates


---
## 4. Extraccion de Features
---

In [26]:
def extract_features(img):
    hist = cv2.calcHist([img], [0], None, [HIST_BINS], [0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    lap = cv2.Laplacian(img, cv2.CV_64F)
    lap = np.abs(lap).astype(np.uint8)
    lap_hist = cv2.calcHist([lap], [0], None, [LAPLACIAN_BINS], [0, 256])
    lap_hist = cv2.normalize(lap_hist, lap_hist).flatten()
    return np.concatenate([hist, lap_hist]).astype(np.float32)

print("Feature extractor listo (64 dims)")

Feature extractor listo (64 dims)


---
## 5. Cargar y procesar todas las imagenes
---

In [29]:
def detect_feature_type(path):
    known = ["valor_", "animal_", "ir_", "personaje_", "serie_a"]
    for prefix in known:
        if path.stem.startswith(prefix):
            end = path.stem.find('_', len(prefix))
            return path.stem[:end] if end != -1 else prefix.rstrip('_')
    for part in path.parts:
        for prefix in known:
            if part.startswith(prefix):
                # Convertir valor_ir_X -> ir_X
                ft = part
                if ft.startswith('valor_ir_'):
                    ft = ft.replace('valor_ir_', 'ir_')
                return ft
    return None

# Cargar features (solo autenticas, sin serie_a)
X = []
stats = Counter()
skip_count = 0

all_images = sorted(SEGMENTS_DIR.rglob("*.png"))
print(f"Procesando {len(all_images)} imagenes...")

for i, img_path in enumerate(all_images):
    ft = detect_feature_type(img_path)
    if ft is None or ft == "serie_a":
        skip_count += 1
        continue
    
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        skip_count += 1
        continue
    
    aligned = aligner.align(img, ft)
    if aligned is None:
        skip_count += 1
        continue
    
    vec = extract_features(aligned)
    X.append(vec)
    stats[ft] += 1
    
    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{len(all_images)}")

X = np.array(X, dtype=np.float32)
print(f"\n[DATASET] {X.shape[0]} muestras, {X.shape[1]} dims (saltados: {skip_count})")
print(f"\nPor tipo de feature:")
for ft, n in sorted(stats.items()):
    print(f"  {ft}: {n}")

Procesando 3166 imagenes...
  500/3166
  1000/3166
  2500/3166
  3000/3166

[DATASET] 2260 muestras, 64 dims (saltados: 906)

Por tipo de feature:
  animal_100bs: 90
  animal_10bs: 90
  animal_200bs: 91
  animal_20bs: 90
  animal_50bs: 90
  ir_100bs: 180
  ir_10bs: 180
  ir_200bs: 182
  ir_20bs: 180
  ir_50bs: 181
  personaje_100bs: 90
  personaje_10bs: 90
  personaje_200bs: 91
  personaje_20bs: 90
  personaje_50bs: 91
  valor_100bs: 91
  valor_10bs: 90
  valor_200bs: 90
  valor_20bs: 90
  valor_50bs: 93


---
## 6. Entrenar One-Class SVM
---

In [30]:
gamma = 1.0 / X.shape[1] if SVM_GAMMA == 'auto' else SVM_GAMMA
print(f"Entrenando One-Class SVM (nu={SVM_NU}, gamma={gamma:.4f})...")

svm = cv2.ml.SVM_create()
svm.setType(cv2.ml.SVM_ONE_CLASS)
svm.setKernel(cv2.ml.SVM_RBF)
svm.setGamma(gamma)
svm.setNu(SVM_NU)

# Para One-Class SVM, todas las etiquetas son 1 (clase positiva = autentica)
labels = np.ones(len(X), dtype=np.int32)
svm.train(X, cv2.ml.ROW_SAMPLE, labels)

print("Entrenamiento completo!")

# Evaluar: cuantas muestras de entrenamiento pasan como autenticas
_ret, y_pred = svm.predict(X)
y_pred = y_pred.flatten()
n_inliers = np.sum(y_pred > 0)
n_outliers = np.sum(y_pred <= 0)
print(f"\n[EVALUACION]")
print(f"  Inliers (OK):  {n_inliers}/{len(X)} ({100*n_inliers/len(X):.1f}%)")
print(f"  Outliers (FAIL): {n_outliers}/{len(X)} ({100*n_outliers/len(X):.1f}%)")
print(f"  Expected: {(1-SVM_NU)*100:.0f}% inliers")

Entrenando One-Class SVM (nu=0.05, gamma=0.0156)...
Entrenamiento completo!

[EVALUACION]
  Inliers (OK):  2148/2260 (95.0%)
  Outliers (FAIL): 112/2260 (5.0%)
  Expected: 95% inliers


---
## 7. Analisis por tipo de feature
---

In [32]:
# Evaluar por tipo de feature
idx = 0
print("Inliers por tipo de feature:")
for ft in sorted(stats.keys()):
    n = stats[ft]
    chunk = y_pred[idx:idx + n]
    ok = np.sum(chunk > 0)
    fail = np.sum(chunk <= 0)
    bar = '#' * int(20 * ok / n) + '-' * int(20 * fail / n)
    print(f"  {ft:20s}: {ok:3d}/{n:3d} OK  [{bar}]")
    idx += n

Inliers por tipo de feature:
  animal_100bs        :  90/ 90 OK  [####################]
  animal_10bs         :  87/ 90 OK  [###################]
  animal_200bs        :  68/ 91 OK  [##############-----]
  animal_20bs         :  77/ 90 OK  [#################--]
  animal_50bs         :  90/ 90 OK  [####################]
  ir_100bs            : 163/180 OK  [##################-]
  ir_10bs             : 175/180 OK  [###################]
  ir_200bs            : 178/182 OK  [###################]
  ir_20bs             : 180/180 OK  [####################]
  ir_50bs             : 177/181 OK  [###################]
  personaje_100bs     :  90/ 90 OK  [####################]
  personaje_10bs      :  90/ 90 OK  [####################]
  personaje_200bs     :  88/ 91 OK  [###################]
  personaje_20bs      :  78/ 90 OK  [#################--]
  personaje_50bs      :  91/ 91 OK  [####################]
  valor_100bs         :  91/ 91 OK  [####################]
  valor_10bs          :  89/ 90 OK  

---
## 8. Exportar modelo
---

In [33]:
OUTPUT_MODEL.parent.mkdir(parents=True, exist_ok=True)
svm.save(str(OUTPUT_MODEL))
print(f"[MODELO] Exportado a {OUTPUT_MODEL}")
print(f"  Tamanio: {OUTPUT_MODEL.stat().st_size / 1024:.1f} KB")
print(f"\nPara usar en la RPi:")
print(f"  svm = cv2.ml.SVM_load('{OUTPUT_MODEL.name}')")
print(f"  _, pred = svm.predict(feature_vector)")
print(f"  autentico = pred[0][0] == 1")

[MODELO] Exportado a ../rpi_deployment/ocsvm.xml
  Tamanio: 70.9 KB

Para usar en la RPi:
  svm = cv2.ml.SVM_load('ocsvm.xml')
  _, pred = svm.predict(feature_vector)
  autentico = pred[0][0] == 1


---
## 9. Prueba con ruido (simulando un falso)
---

In [9]:
# Probar que el modelo rechaza datos aleatorios
noise = np.random.rand(10, 64).astype(np.float32)
_ret, pred_noise = svm.predict(noise)
rejected = np.sum(pred_noise.flatten() <= 0)
print(f"Ruido aleatorio rechazado: {rejected}/10 (deberia ser 10/10)")

# Probar con una muestra real
sample = X[0:1]
_ret, pred_sample = svm.predict(sample)
print(f"Muestra real aceptada: {pred_sample[0][0] > 0}")

Ruido aleatorio rechazado: 0/10 (deberia ser 10/10)
Muestra real aceptada: True
